# Notebook 16 - FM input-scope and pooling ablation

Compare six options for fixed configuration B: token windows with radius 4 (9 tokens per allele), radius 8 (17 tokens per allele), and radius 16 (33 tokens per allele; reused from Step 2), plus pooled FM `center`, `mean`, and `CLS` variants. Full-token training is disabled. The FM pair is fixed to NTv1 DNA 1001 + ESM1b protein 501; features are bio_core + SpliceAI + geometry k=64; seeds are 5, 6, 7, 37, and 42.

Radius 4/8 and pooled center/mean/CLS are trained with the saved Step 2 hyperparameters for each seed and the same three-modality Cross-Attention + MLP. Mean pooling after Cross-Attention is fixed. Existing token embeddings are reused; FM inference is not rerun. Checkpoints, epoch history, metrics, profiling, and resume files are stored under `step3_pooling_ablation`.


In [1]:
from pathlib import Path
import importlib, json, sys
import pandas as pd
from IPython.display import display

_cwd = Path.cwd().resolve()
REPO_ROOT = next((p for p in (_cwd, *_cwd.parents) if (p / "core").is_dir() and (p / "controllers").is_dir()), _cwd)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

BASE_DIR = Path(r"F:\variant_data")
BATCH_RUN_DIR = REPO_ROOT / "experiments" / "fm_ablation_20261117_003149"
MODULE06_ROOT = BATCH_RUN_DIR / "module06_architecture_experiments"
STEP2_ROOT = MODULE06_ROOT / "step2_sequence_length_ablation"
STEP2_RUNS_ROOT = STEP2_ROOT / "runs"
STEP3_ROOT = MODULE06_ROOT / "step3_pooling_ablation"
STEP3_ROOT.mkdir(parents=True, exist_ok=True)
TOKEN_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "token_level_cache" / "module06"
POOLED_VECTOR_CACHE_ROOT = BASE_DIR / "fm_embeddings" / "step3_pooling_ablation"
GEOMETRY_ROOT_NAME = "geometry_v2_k64_step2_matched"
GEOMETRY_NORMALIZATION_ROOT = "normalization_artifacts_v2_step2_matched"
GEOMETRY_INDEX_ROOT = "faiss_indexes_v2_step2_matched"
GEOMETRY_K = 64
SEEDS = (5, 6, 7, 37, 42)
GEOM_FEATURE_NAMES = [
    f"{modality}_{feature}"
    for modality in ("dna", "prot")
    for feature in ("LVD_L2", "LVD_Cosine", "LID", "LVD_Relative")
]
TARGET_CONFIGS = [{
    "name": "B_with_geometry",
    "ablation": "bio_core_dna_prot_spliceai_geom",
    "active_modalities": ["dna", "prot", "bio_core", "spliceai", "geom"],
    "geom_feature_names": GEOM_FEATURE_NAMES,
    "geometry_root": GEOMETRY_ROOT_NAME,
}]
DATASETS = [
    {"name": "Train3Val_Test", "train": "train3", "val": "val", "test": "test"},
    {"name": "Train3Val_ClinVarHQ", "train": "train3", "val": "val", "test": "clinvarhq"},
    {"name": "Train3Val_ProteinGym", "train": "train3", "val": "val", "test": "proteingym"},
    {"name": "Train3Val_UniProt", "train": "train3", "val": "val", "test": "uniprot"},
]

from core.module06_architecture_experiments.fm_ablation_preparation import build_split_configs
from core.module06_architecture_experiments.fm_sequence_length_ablation import build_selected_model_configs
from core.module06_architecture_experiments.pooling_ablation_experiment import (
    load_step2_seed_configs,
)

SPLITS = build_split_configs(BASE_DIR)
MODEL_CONFIGS = build_selected_model_configs(dna_lengths=(1001,), protein_lengths=(501,))
DNA_MODEL, PROTEIN_MODEL = MODEL_CONFIGS
if DNA_MODEL["name"] != "nt_v1_500m_dna_1001" or PROTEIN_MODEL["name"] != "esm1b_650m_prot_501":
    raise ValueError("Notebook 16 is fixed to NTv1 DNA 1001 + ESM1b protein 501.")
TOKEN_CACHE_INDEX_PATH = MODULE06_ROOT / "fm_ablation_token_cache_index.json"
if not TOKEN_CACHE_INDEX_PATH.is_file():
    raise FileNotFoundError(f"Step 2 token-cache index is missing: {TOKEN_CACHE_INDEX_PATH}")
TOKEN_CACHE_INDEX = json.loads(TOKEN_CACHE_INDEX_PATH.read_text(encoding="utf-8"))
STEP2_CONFIGS, STEP2_MANIFESTS = load_step2_seed_configs(STEP2_RUNS_ROOT, SEEDS)
STEP2_PIPELINE_CONFIG = json.loads(
    Path(STEP2_MANIFESTS[5]["manifest"]).read_text(encoding="utf-8")
)["config"]["pipeline_config"]

for _model in MODEL_CONFIGS:
    for _split in SPLITS:
        _entry = TOKEN_CACHE_INDEX["models"][_model["name"]][_split["name"]]
        if _entry.get("status") != "complete":
            raise RuntimeError(f"Token cache is not complete: {_model['name']}/{_split['name']}")
        _geom = (
            BASE_DIR / GEOMETRY_ROOT_NAME / "train3" / _split["name"]
            / f"{_model['name']}_center_geom_v2_norm.parquet"
        )
        if not _geom.is_file():
            raise FileNotFoundError(f"Matched Step 2 geometry file is missing: {_geom}")

print("Batch:", BATCH_RUN_DIR)
print("Models:", DNA_MODEL["name"], "+", PROTEIN_MODEL["name"])
print("Fixed target:", TARGET_CONFIGS[0]["name"], "| features: bio_core + SpliceAI + geometry k=64")
print("Seeds:", SEEDS)
print("Step 2 checkpoint cohort:", {seed: STEP2_MANIFESTS[seed]["checkpoint"] for seed in SEEDS})
print("Output root:", STEP3_ROOT)

c:\Users\Dung\anaconda3\envs\missense_variant_patho_predict_clean\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Batch: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149
Models: nt_v1_500m_dna_1001 + esm1b_650m_prot_501
Fixed target: B_with_geometry | features: bio_core + SpliceAI + geometry k=64
Seeds: (5, 6, 7, 37, 42)
Step 2 checkpoint cohort: {5: 'D:\\Missense_variant_pathogenicity_prediction\\experiments\\fm_ablation_20261117_003149\\module06_architecture_experiments\\step2_sequence_length_ablation\\runs\\step2_nt_v1_dna1001_esm1b_prot501_seed5\\checkpoints\\B_with_geometry\\best_model.pth', 6: 'D:\\Missense_variant_pathogenicity_prediction\\experiments\\fm_ablation_20261117_003149\\module06_architecture_experiments\\step2_sequence_length_ablation\\runs\\step2_nt_v1_dna1001_esm1b_prot501_seed6\\checkpoints\\B_with_geometry\\best_model.pth', 7: 'D:\\Missense_variant_pathogenicity_prediction\\experiments\\fm_ablation_20261117_003149\\module06_architecture_experiments\\step2_sequence_length_ablation\\runs\\step2_nt_v1_dna1001_esm1b_prot501_seed7\\checkpoints\\

## Prepare pooled FM vectors

Center, mean and CLS vectors are derived from the existing full-token cache and saved in a separate resumable cache. No new FM inference runs. The `mean` definition matches Module 2 and includes all valid attention-mask positions, including valid special tokens.

In [2]:
from core.module06_architecture_experiments.pooling_ablation_experiment import (
    build_pooled_vectors_from_token_cache,
)

STEP3_POOLED_VECTOR_INDEX = build_pooled_vectors_from_token_cache(
    token_cache_index=TOKEN_CACHE_INDEX,
    model_configs=MODEL_CONFIGS,
    split_configs=SPLITS,
    output_root=POOLED_VECTOR_CACHE_ROOT,
    poolings=("center", "mean", "cls"),
)
for _model in MODEL_CONFIGS:
    for _split in SPLITS:
        _modes = STEP3_POOLED_VECTOR_INDEX["models"][_model["name"]][_split["name"]]
        if any(_modes[_pool]["status"] != "complete" for _pool in ("center", "mean", "cls")):
            raise RuntimeError(f"Pooled cache incomplete: {_model['name']}/{_split['name']}")
print("All pooled FM vectors are ready/resumable:", POOLED_VECTOR_CACHE_ROOT)

[POOL CACHE RESUME] nt_v1_500m_dna_1001/train3
[POOL CACHE RESUME] nt_v1_500m_dna_1001/val
[POOL CACHE RESUME] nt_v1_500m_dna_1001/test
[POOL CACHE RESUME] nt_v1_500m_dna_1001/clinvarhq
[POOL CACHE RESUME] nt_v1_500m_dna_1001/proteingym
[POOL CACHE RESUME] nt_v1_500m_dna_1001/uniprot
[POOL CACHE RESUME] esm1b_650m_prot_501/train3
[POOL CACHE RESUME] esm1b_650m_prot_501/val
[POOL CACHE RESUME] esm1b_650m_prot_501/test
[POOL CACHE RESUME] esm1b_650m_prot_501/clinvarhq
[POOL CACHE RESUME] esm1b_650m_prot_501/proteingym
[POOL CACHE RESUME] esm1b_650m_prot_501/uniprot
All pooled FM vectors are ready/resumable: F:\variant_data\fm_embeddings\step3_pooling_ablation


## Train radius 4/8 and pooled variants; reuse Step 2 radius 16

Train token windows `k_radius=4/8` and pooled `center`, `mean`, `CLS`, with batch size 64. Read the Step 2 `k_radius=16` checkpoint and metrics directly for comparison. Each trained variant uses its seed-specific saved Step 2 settings, resumes from its own checkpoint, and skips completed test evaluations.


In [3]:
from core.module05v2_fusion_classifier.pipeline import FusionBatchPipeline
from core.module06_architecture_experiments.pooling_ablation_experiment import (
    run_step3_pooling_ablation,
)

FM_PROFILE_JSON = BASE_DIR / "profiling" / "fm_profiling_sequence_ablation.json"

def _step3_pipeline_factory(seed, model_configs):
    return FusionBatchPipeline(
        base_dir=str(BASE_DIR),
        config={**STEP2_PIPELINE_CONFIG, "seed": int(seed)},
        datasets=DATASETS,
        models_space=model_configs,
        pooling_strategies=["center"],
        experiments=[{
            "name": "PyTorch_CrossAttention_PoolingAblation",
            "type": "pytorch",
            "fusion": "cross_attention",
        }],
        explainability={"enable_shap": False, "enable_lime": False},
        fm_profile_json=str(FM_PROFILE_JSON),
        geom_profile_json=None,
        geometry_dir=str(BASE_DIR / GEOMETRY_ROOT_NAME / "{context}"),
        geom_feature_names=GEOM_FEATURE_NAMES,
        batch_run_dir=str(BATCH_RUN_DIR),
        embedding_dir=str(BASE_DIR / "fm_embeddings" / "sequence_ablation"),
    )

# Reload the shared runner and pooled-vector adapter.
importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.architecture_experiment_runner"
))
pooling_ablation_module = importlib.reload(importlib.import_module(
    "core.module06_architecture_experiments.pooling_ablation_experiment"
))
STEP3_RUN_RESULT = pooling_ablation_module.run_step3_pooling_ablation(
    pipeline_factory=_step3_pipeline_factory,
    target_configs=TARGET_CONFIGS,
    datasets=DATASETS,
    model_configs=MODEL_CONFIGS,
    token_cache_index=TOKEN_CACHE_INDEX,
    pooled_vector_index=STEP3_POOLED_VECTOR_INDEX,
    step2_configs=STEP2_CONFIGS,
    step2_manifests=STEP2_MANIFESTS,
    step_root=STEP3_ROOT,
    seeds=SEEDS,
    device="cuda",
    pooled_batch_size=64,
    display_fn=print,
)
print(STEP3_RUN_RESULT)

[CrossAttention_ThreeModalities_FM_PoolingAblation] Preparing train dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 23208 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/train3: 100%|██████████| 11604/11604 [02:50<00:00, 68.00shard/s] 


[Window cache] dna/train3: 23208 samples ready in 170.7s


Window cache protein/train3: 100%|██████████| 1451/1451 [00:35<00:00, 40.86shard/s]


[Window cache] protein/train3: 23208 samples ready in 35.5s
[CrossAttention_ThreeModalities_FM_PoolingAblation] Train loader ready: 363 batches in 207.9s
[CrossAttention_ThreeModalities_FM_PoolingAblation] Preparing validation dataset and token-cache loader...
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 13701 mẫu. Đã tối ưu hóa RAM & CPU 100%.



Window cache dna/val: 100%|██████████| 6851/6851 [01:00<00:00, 113.97shard/s]


... (532 outputs left out)

[CrossAttention_ThreeModalities_FM_PoolingAblation] Test complete: B_with_geometry / Train3Val_Test MCC=0.7588 AUROC=0.9497 AUPRC=0.8990
[CrossAttention_ThreeModalities_FM_PoolingAblation] Evaluating test set: B_with_geometry / Train3Val_ClinVarHQ
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 703 mẫu. Đã tối ưu hóa RAM & CPU 100%.



[CrossAttention_ThreeModalities_FM_PoolingAblation] Test complete: B_with_geometry / Train3Val_ClinVarHQ MCC=0.8141 AUROC=0.9804 AUPRC=0.9832
[CrossAttention_ThreeModalities_FM_PoolingAblation] Evaluating test set: B_with_geometry / Train3Val_ProteinGym
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 1472 mẫu. Đã tối ưu hóa RAM & CPU 100%.



[CrossAttention_ThreeModalities_FM_PoolingAblation] Test complete: B_with_geometry / Train3Val_ProteinGym MCC=0.6116 AUROC=0.8865 AUPRC=0.8681
[CrossAttention_ThreeModalities_FM_PoolingAblation] Evaluating test set: B_with_geometry / Train3Val_UniProt
[*] Đang nạp dữ liệu bảng (Backbone). Active Mods: ['bio_core', 'spliceai', 'geom']
[+] Dataset sẵn sàng: 1333 mẫu. Đã tối ưu hóa RAM & CPU 100%.



[CrossAttention_ThreeModalities_FM_PoolingAblation] Test complete: B_with_geometry / Train3Val_UniProt MCC=0.4244 AUROC=0.7501 AUPRC=0.7348
CrossAttention_ThreeModalities_FM_PoolingAblation training, validation, and test evaluation complete. Results and checkpoints: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\step3_pooling_ablation\runs\step3_pooling_pooled_cls_seed42
     Target_Config               Dataset  \
1  B_with_geometry   Train3Val_ClinVarHQ   
2  B_with_geometry  Train3Val_ProteinGym   
0  B_with_geometry        Train3Val_Test   
3  B_with_geometry     Train3Val_UniProt   

                                        Architecture  \
1  CrossAttention_ThreeModalities_FM_PoolingAblation   
2  CrossAttention_ThreeModalities_FM_PoolingAblation   
0  CrossAttention_ThreeModalities_FM_PoolingAblation   
3  CrossAttention_ThreeModalities_FM_PoolingAblation   

                     Variant  Decision_Threshold  Ac

## Results: per seed and mean/std by test set

`window_radius16_len33` is reused from Step 2; `window_radius4_len9`, `window_radius8_len17`, and pooled `center`, `mean`, `CLS` come from Step 3. Results are shown by seed and as mean/std for validation and each test set. Test metrics are reports, not model-selection criteria.


In [4]:
from core.module06_architecture_experiments.pooling_ablation_experiment import (
    summarize_step3_pooling_ablation,
)

STEP3_SUMMARY = summarize_step3_pooling_ablation(
    step2_runs_root=STEP2_RUNS_ROOT,
    step3_root=STEP3_ROOT,
    seeds=SEEDS,
)
STEP3_PROFILE_COLUMNS = [
    "FM_GFLOPs/Sample", "FM_Latency_ms/Sample", "Geom_GFLOPs/Sample",
    "Geom_Latency_ms/Sample", "Classifier_GFLOPs/Sample",
    "Classifier_Latency_ms/Sample", "Classifier_Total_Params",
    "Classifier_Param_Mem_MB", "Classifier_Peak_VRAM_MB",
    "E2E_GFLOPs/Sample", "E2E_Latency_ms/Sample", "E2E_Total_Params",
    "E2E_Param_Mem_MB", "Max_Peak_VRAM_MB",
]
STEP3_CLASSIFICATION_METRICS = [
    "Accuracy", "Precision", "Recall", "Specificity", "F1_Score", "MCC", "AUROC", "AUPRC",
]
STEP3_VALIDATION_METRICS = [f"Val_{name}" for name in STEP3_CLASSIFICATION_METRICS]

print("Validation results by seed:")
_STEP3_VAL_COLS = [
    "Pooling_Ablation_Mode", "Seed", "Target_Config", *STEP3_VALIDATION_METRICS,
    "Decision_Threshold", "Best_Epoch", "Fit_Seconds", "Checkpoint_Path",
]
display(STEP3_SUMMARY["validation_by_seed"][[
    c for c in _STEP3_VAL_COLS if c in STEP3_SUMMARY["validation_by_seed"].columns
]].sort_values(["Val_MCC", "Pooling_Ablation_Mode"], ascending=[False, True]).reset_index(drop=True))
print("Validation mean/std across seeds (ranked by mean validation MCC):")
display(STEP3_SUMMARY["validation_mean_std"])

for _dataset, _rows in STEP3_SUMMARY["test_by_seed"].groupby("Dataset", sort=False):
    print(f"Test results by seed: {_dataset}")
    _seed_cols = [
        "Pooling_Ablation_Mode", "Seed", "Target_Config", *STEP3_CLASSIFICATION_METRICS,
        *STEP3_PROFILE_COLUMNS, "Decision_Threshold", "Best_Epoch", "Fit_Seconds",
        "Checkpoint_Path",
    ]
    display(_rows[[c for c in _seed_cols if c in _rows.columns]].sort_values(
        ["MCC", "Pooling_Ablation_Mode"], ascending=[False, True]
    ).reset_index(drop=True))
    print(f"Mean/std across seeds: {_dataset} (ranked by mean MCC)")
    _summary_rows = STEP3_SUMMARY["test_mean_std_by_dataset"].query("Dataset == @_dataset")
    _summary_cols = ["Pooling_Ablation_Mode", "Target_Config", "N_Seeds"]
    _summary_cols += [f"{metric}_{stat}" for metric in [*STEP3_CLASSIFICATION_METRICS, *STEP3_PROFILE_COLUMNS]
                      for stat in ("mean", "std")]
    display(_summary_rows[[c for c in _summary_cols if c in _summary_rows.columns]].sort_values(
        "MCC_mean", ascending=False
    ).reset_index(drop=True))
print("Saved per-seed and mean/std CSVs under:", STEP3_SUMMARY["output_root"])


Validation results by seed:


,Pooling_Ablation_Mode,Seed,Target_Config,Val_Accuracy,Val_Precision,Val_Recall,Val_Specificity,Val_F1_Score,Val_MCC,Val_AUROC,Val_AUPRC,Decision_Threshold,Best_Epoch,Fit_Seconds,Checkpoint_Path
0,window_radius8_len17,7,B_with_geometry,0.9463,0.8637,0.8531,0.9683,0.8584,0.8252,0.9756,0.9222,0.541660,2,95.734987,D:\Missense_variant_pathogenicity_prediction\e...
1,window_radius16_len33,37,B_with_geometry,0.9458,0.8597,0.8558,0.9671,0.8577,0.8243,0.9755,0.9237,0.479132,2,144.433141,D:\Missense_variant_pathogenicity_prediction\e...
2,window_radius4_len9,7,B_with_geometry,0.9456,0.8568,0.8585,0.9662,0.8576,0.8240,0.9755,0.9210,0.518015,2,88.241248,D:\Missense_variant_pathogenicity_prediction\e...
3,window_radius16_len33,6,B_with_geometry,0.9463,0.8726,0.8412,0.9710,0.8566,0.8238,0.9755,0.9225,0.918221,4,143.026107,D:\Missense_variant_pathogenicity_prediction\e...
4,window_radius16_len33,7,B_with_geometry,0.9460,0.8675,0.8462,0.9695,0.8567,0.8235,0.9755,0.9226,0.559536,2,144.708570,D:\Missense_variant_pathogenicity_prediction\e...
5,window_radius16_len33,42,B_with_geometry,0.9445,0.8514,0.8592,0.9646,0.8553,0.8210,0.9751,0.9213,0.814246,2,119.053869,D:\Missense_variant_pathogenicity_prediction\e...
6,window_radius16_len33,5,B_with_geometry,0.9456,0.8764,0.8324,0.9723,0.8538,0.8209,0.9756,0.9229,0.888212,2,120.806752,D:\Missense_variant_pathogenicity_prediction\e...
7,pooled_center,37,B_with_geometry,0.9445,0.8511,0.8592,0.9646,0.8551,0.8208,0.9749,0.9198,0.446996,2,81.596757,D:\Missense_variant_pathogenicity_prediction\e...
8,window_radius8_len17,37,B_with_geometry,0.9445,0.8693,0.8344,0.9704,0.8515,0.8176,0.9744,0.9181,0.523351,2,95.897042,D:\Missense_variant_pathogenicity_prediction\e...
9,window_radius4_len9,6,B_with_geometry,0.9441,0.8629,0.8405,0.9685,0.8516,0.8172,0.9744,0.9194,0.824203,2,88.984080,D:\Missense_variant_pathogenicity_prediction\e...


Validation mean/std across seeds (ranked by mean validation MCC):


,Pooling_Ablation_Mode,Target_Config,N_Seeds_mean,N_Seeds_std,Best_Epoch_mean,Best_Epoch_std,Fit_Seconds_mean,Fit_Seconds_std,Decision_Threshold_mean,Decision_Threshold_std,...,Val_Recall_std,Val_Specificity_mean,Val_Specificity_std,Val_F1_Score_mean,Val_F1_Score_std,Val_AUROC_mean,Val_AUROC_std,Val_AUPRC_mean,Val_AUPRC_std,N_Seeds
3,window_radius16_len33,B_with_geometry,1.0,0.0,2.4,0.894427,134.405688,13.244057,0.731869,0.199707,...,0.010882,0.96890,0.003085,0.85602,0.001506,0.97544,0.000195,0.92260,0.000866,5
5,window_radius8_len17,B_with_geometry,1.0,0.0,1.6,0.547723,90.928408,6.530026,0.693616,0.157810,...,0.014507,0.97082,0.002744,0.85086,0.004572,0.97386,0.001429,0.91780,0.002815,5
4,window_radius4_len9,B_with_geometry,1.0,0.0,2.4,0.894427,92.786902,9.329500,0.683261,0.254480,...,0.018784,0.96780,0.003632,0.85056,0.005230,0.97372,0.001720,0.91756,0.002982,5
0,pooled_center,B_with_geometry,1.0,0.0,1.8,1.303840,78.687539,12.725861,0.699685,0.158144,...,0.012073,0.96724,0.002154,0.84876,0.004258,0.97408,0.000540,0.91570,0.005105,5
2,pooled_mean,B_with_geometry,1.0,0.0,6.0,2.828427,121.276141,28.412169,0.756340,0.062386,...,0.008088,0.95544,0.003862,0.81482,0.003769,0.96386,0.000945,0.87822,0.005712,5
1,pooled_cls,B_with_geometry,1.0,0.0,2.2,0.447214,84.358150,4.790741,0.632741,0.176168,...,0.027566,0.95778,0.008316,0.81332,0.002406,0.96350,0.001079,0.88498,0.000968,5


Test results by seed: Train3Val_Test


,Pooling_Ablation_Mode,Seed,Target_Config,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Decision_Threshold,Best_Epoch,Fit_Seconds,Checkpoint_Path
0,window_radius16_len33,37,B_with_geometry,0.9377,0.8868,0.8836,0.9579,0.8852,0.8424,0.9736,...,643.05,820.8911,34.67,1171332260,2297.61,2132.94,0.479132,2,144.433141,D:\Missense_variant_pathogenicity_prediction\e...
1,window_radius4_len9,7,B_with_geometry,0.9365,0.8826,0.8842,0.9561,0.8834,0.8398,0.9742,...,463.53,820.1578,34.59,1171327649,2297.59,2132.94,0.518015,2,88.241248,D:\Missense_variant_pathogenicity_prediction\e...
2,window_radius16_len33,7,B_with_geometry,0.9367,0.8888,0.8770,0.9590,0.8828,0.8395,0.9751,...,639.46,820.8911,34.66,1171332260,2297.61,2132.94,0.559536,2,144.708570,D:\Missense_variant_pathogenicity_prediction\e...
3,window_radius16_len33,5,B_with_geometry,0.9367,0.9030,0.8595,0.9655,0.8807,0.8381,0.9734,...,638.23,820.8911,34.64,1171332260,2297.61,2132.94,0.888212,2,120.806752,D:\Missense_variant_pathogenicity_prediction\e...
4,pooled_center,37,B_with_geometry,0.9358,0.8842,0.8794,0.9570,0.8818,0.8378,0.9741,...,464.27,819.9133,34.59,1172425377,2301.78,2132.94,0.446996,2,81.596757,D:\Missense_variant_pathogenicity_prediction\e...
5,window_radius4_len9,6,B_with_geometry,0.9358,0.8918,0.8697,0.9606,0.8806,0.8369,0.9734,...,465.31,820.1578,34.59,1171327649,2297.59,2132.94,0.824203,2,88.984080,D:\Missense_variant_pathogenicity_prediction\e...
6,window_radius8_len17,6,B_with_geometry,0.9352,0.8954,0.8625,0.9624,0.8786,0.8347,0.9734,...,479.10,820.4022,34.60,1171327649,2297.59,2132.94,0.786734,2,95.449192,D:\Missense_variant_pathogenicity_prediction\e...
7,window_radius4_len9,37,B_with_geometry,0.9345,0.8804,0.8788,0.9554,0.8796,0.8346,0.9730,...,465.31,820.1578,34.60,1171327649,2297.59,2132.94,0.315725,2,88.380303,D:\Missense_variant_pathogenicity_prediction\e...
8,window_radius16_len33,42,B_with_geometry,0.9347,0.8846,0.8739,0.9574,0.8792,0.8345,0.9713,...,641.36,820.8911,34.64,1171332260,2297.61,2132.94,0.814246,2,119.053869,D:\Missense_variant_pathogenicity_prediction\e...
9,window_radius8_len17,37,B_with_geometry,0.9344,0.8981,0.8559,0.9637,0.8765,0.8323,0.9739,...,479.10,820.4022,34.61,1171327649,2297.59,2132.94,0.523351,2,95.897042,D:\Missense_variant_pathogenicity_prediction\e...


... (11 outputs left out)

,Pooling_Ablation_Mode,Seed,Target_Config,Accuracy,Precision,Recall,Specificity,F1_Score,MCC,AUROC,...,Classifier_Peak_VRAM_MB,E2E_GFLOPs/Sample,E2E_Latency_ms/Sample,E2E_Total_Params,E2E_Param_Mem_MB,Max_Peak_VRAM_MB,Decision_Threshold,Best_Epoch,Fit_Seconds,Checkpoint_Path
0,window_radius16_len33,7,B_with_geometry,0.7802,0.7685,0.7997,0.7608,0.7838,0.5609,0.8225,...,639.46,820.8911,35.14,1171332260,2297.61,2132.94,0.559536,2,144.708570,D:\Missense_variant_pathogenicity_prediction\e...
1,window_radius8_len17,5,B_with_geometry,0.7764,0.7773,0.7726,0.7803,0.7749,0.5529,0.8190,...,479.10,820.4022,35.14,1171327649,2297.59,2132.94,0.887605,1,83.507332,D:\Missense_variant_pathogenicity_prediction\e...
2,window_radius8_len17,7,B_with_geometry,0.7742,0.7627,0.7937,0.7549,0.7779,0.5489,0.8198,...,479.10,820.4022,35.07,1171327649,2297.59,2132.94,0.541660,2,95.734987,D:\Missense_variant_pathogenicity_prediction\e...
3,pooled_center,37,B_with_geometry,0.7734,0.7616,0.7937,0.7534,0.7773,0.5474,0.8223,...,464.27,819.9133,35.07,1172425377,2301.78,2132.94,0.446996,2,81.596757,D:\Missense_variant_pathogenicity_prediction\e...
4,window_radius4_len9,6,B_with_geometry,0.7727,0.7658,0.7831,0.7623,0.7744,0.5456,0.8170,...,465.31,820.1578,35.08,1171327649,2297.59,2132.94,0.824203,2,88.984080,D:\Missense_variant_pathogenicity_prediction\e...
5,pooled_center,6,B_with_geometry,0.7719,0.7671,0.7786,0.7653,0.7728,0.5440,0.8181,...,464.27,819.9133,35.04,1172425377,2301.78,2132.94,0.737753,4,99.804696,D:\Missense_variant_pathogenicity_prediction\e...
6,window_radius4_len9,7,B_with_geometry,0.7712,0.7553,0.7997,0.7429,0.7769,0.5434,0.8179,...,463.53,820.1578,35.07,1171327649,2297.59,2132.94,0.518015,2,88.241248,D:\Missense_variant_pathogenicity_prediction\e...
7,window_radius16_len33,5,B_with_geometry,0.7697,0.7725,0.7620,0.7773,0.7672,0.5394,0.8171,...,638.23,820.8911,35.11,1171332260,2297.61,2132.94,0.888212,2,120.806752,D:\Missense_variant_pathogenicity_prediction\e...
8,window_radius16_len33,37,B_with_geometry,0.7689,0.7557,0.7922,0.7459,0.7735,0.5386,0.8164,...,643.05,820.8911,35.14,1171332260,2297.61,2132.94,0.479132,2,144.433141,D:\Missense_variant_pathogenicity_prediction\e...
9,pooled_center,5,B_with_geometry,0.7682,0.7645,0.7726,0.7638,0.7685,0.5364,0.8121,...,464.27,819.9133,35.06,1172425377,2301.78,2132.94,0.884186,1,69.916331,D:\Missense_variant_pathogenicity_prediction\e...


Mean/std across seeds: Train3Val_UniProt (ranked by mean MCC)


,Pooling_Ablation_Mode,Target_Config,N_Seeds,Accuracy_mean,Accuracy_std,Precision_mean,Precision_std,Recall_mean,Recall_std,Specificity_mean,...,E2E_GFLOPs/Sample_mean,E2E_GFLOPs/Sample_std,E2E_Latency_ms/Sample_mean,E2E_Latency_ms/Sample_std,E2E_Total_Params_mean,E2E_Total_Params_std,E2E_Param_Mem_MB_mean,E2E_Param_Mem_MB_std,Max_Peak_VRAM_MB_mean,Max_Peak_VRAM_MB_std
0,window_radius16_len33,B_with_geometry,5,0.76728,0.008967,0.76136,0.008520,0.77620,0.018729,0.75844,...,820.8911,0.0,35.124,0.015166,1.171332e+09,0.0,2297.61,0.0,2132.94,0.0
1,window_radius8_len17,B_with_geometry,5,0.76712,0.009995,0.76564,0.006618,0.76748,0.023603,0.76682,...,820.4022,0.0,35.090,0.028284,1.171328e+09,0.0,2297.59,0.0,2132.94,0.0
2,window_radius4_len9,B_with_geometry,5,0.76458,0.007660,0.76028,0.008257,0.77078,0.031345,0.75844,...,820.1578,0.0,35.096,0.047223,1.171328e+09,0.0,2297.59,0.0,2132.94,0.0
3,pooled_center,B_with_geometry,5,0.76142,0.013788,0.75676,0.010675,0.76776,0.023092,0.75516,...,819.9133,0.0,35.054,0.011402,1.172425e+09,0.0,2301.78,0.0,2132.94,0.0
4,pooled_mean,B_with_geometry,5,0.72406,0.006339,0.71518,0.007668,0.74156,0.017430,0.70674,...,819.9133,0.0,35.056,0.008944,1.172425e+09,0.0,2301.78,0.0,2132.94,0.0
5,pooled_cls,B_with_geometry,5,0.71658,0.009822,0.71514,0.018524,0.71808,0.028238,0.71510,...,819.9133,0.0,35.046,0.008944,1.172425e+09,0.0,2301.78,0.0,2132.94,0.0


Saved per-seed and mean/std CSVs under: D:\Missense_variant_pathogenicity_prediction\experiments\fm_ablation_20261117_003149\module06_architecture_experiments\step3_pooling_ablation
